In [1]:
%pip install nibabel scipy scikit-learn pytorch-msssim -q

import os, json, hashlib, warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import nibabel as nib
from scipy.signal import find_peaks
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from collections import defaultdict
warnings.filterwarnings('ignore')

OUTPUT_DIR = '/kaggle/working/qsyncardiotwin_v2'
for d in ['patients', 'figures', 'tables', 'validation']:
    os.makedirs(f'{OUTPUT_DIR}/{d}', exist_ok=True)

np.random.seed(42)
print("Setup complete.")

Note: you may need to restart the kernel to use updated packages.
Setup complete.


In [2]:
def generate_cine_phantom(phenotype, n_phases=20, img_size=128):
    edv   = phenotype['EDV_mL']
    esv   = phenotype['ESV_mL']
    wt    = phenotype['wall_thickness_mm']
    disease = phenotype.get('disease', 'NORMAL')

    r_lv_ed = float(np.clip((edv * 600 / np.pi) ** (1/3), 14, 32))
    r_lv_es = float(np.clip((esv * 600 / np.pi) ** (1/3), 8, r_lv_ed - 3))
    wt_px   = float(np.clip(wt * 0.75, 3, 8))
    r_rv    = r_lv_ed * 0.72

    cx, cy = img_size // 2, img_size // 2
    cx_rv  = cx + int(r_lv_ed + r_rv * 0.85)
    Y, X   = np.ogrid[:img_size, :img_size]

    phases = []
    for i in range(n_phases):
        t     = i / n_phases
        r_lv  = r_lv_ed - (r_lv_ed - r_lv_es) * np.sin(np.pi * t)
        r_rv_i = r_rv  - (r_rv - r_rv * 0.75) * np.sin(np.pi * t)

        img   = np.zeros((img_size, img_size), dtype=np.float32)
        dist_lv = np.sqrt((X - cx)**2    + (Y - cy)**2)
        dist_rv = np.sqrt((X - cx_rv)**2 + (Y - cy)**2)

        img[(dist_lv >= r_lv) & (dist_lv < r_lv + wt_px)] = 0.70
        img[dist_lv < r_lv]  = 1.00
        img[dist_rv < r_rv_i] = 0.85

        if disease == 'HCM':
            img[(dist_lv >= r_lv) & (dist_lv < r_lv + wt_px * 1.6)] = 0.72

        if disease == 'DCM':
            img[dist_lv < r_lv * 1.15] = 0.90

        if disease == 'INFARCT' and i == 0:
            angle = np.arctan2(Y - cy, X - cx)
            scar  = (dist_lv >= r_lv) & (dist_lv < r_lv + wt_px) \
                    & (angle > np.pi/6) & (angle < np.pi/2)
            img[scar] = 0.20

        img = np.clip(img, 0, 1)
        phases.append(img)

    return np.stack(phases, axis=0)   # (n_phases, H, W)

print("Cine phantom generator ready.")

Cine phantom generator ready.


In [3]:
def generate_ecg_12lead(phenotype, duration_sec=10.0, fs=500):
    hr      = float(phenotype.get('heart_rate_bpm', 70))
    ef      = float(phenotype.get('EF_percent', 60))
    wt      = float(phenotype.get('wall_thickness_mm', 9))
    disease = phenotype.get('disease', 'NORMAL')

    n_samples  = int(duration_sec * fs)
    rr_samples = max(int(60.0 / hr * fs), 100)

    def single_beat(n, disease, ef, wt):
        b = np.zeros(n, dtype=np.float32)
        # P-wave
        p_pos = int(n * 0.14)
        p_amp = 0.14 if disease != 'HCM' else 0.26
        b += p_amp * np.exp(-((np.arange(n)-p_pos)**2) / (2*(n*0.03)**2))
        # Q-wave
        q_pos = int(n * 0.30)
        b -= 0.12 * np.exp(-((np.arange(n)-q_pos)**2) / (2*(n*0.018)**2))
        # R-wave
        r_pos = int(n * 0.35)
        r_amp = 0.95 if disease != 'DCM' else 0.60
        r_amp = r_amp if disease != 'INFARCT' else r_amp * 0.80
        r_w   = n * 0.035 if disease != 'INFARCT' else n * 0.055
        b += r_amp * np.exp(-((np.arange(n)-r_pos)**2) / (2*r_w**2))
        # S-wave
        s_pos = int(n * 0.40)
        b -= 0.08 * np.exp(-((np.arange(n)-s_pos)**2) / (2*(n*0.02)**2))
        # ST segment (infarct: depression)
        if disease == 'INFARCT':
            st_s, st_e = int(n*0.41), int(n*0.58)
            b[st_s:st_e] -= 0.12
        # T-wave
        t_pos = int(n * 0.65)
        t_amp = 0.32 if ef > 50 else 0.18
        t_w   = n * 0.07 if disease != 'HCM' else n * 0.11
        b += t_amp * np.exp(-((np.arange(n)-t_pos)**2) / (2*t_w**2))
        return b

    beat_tmpl = single_beat(rr_samples, disease, ef, wt)
    lead_II = np.zeros(n_samples, dtype=np.float32)
    pos = 0
    while pos + rr_samples <= n_samples:
        lead_II[pos:pos+rr_samples] += beat_tmpl
        pos += rr_samples

    factors = {
        'I':1.0, 'II':1.2, 'III':0.5, 'aVR':-0.85,
        'aVL':0.40, 'aVF':0.70,
        'V1':-0.25, 'V2':0.50, 'V3':0.80,
        'V4':1.05, 'V5':0.95, 'V6':0.72
    }
    ecg_12 = np.stack(
        [lead_II * f + np.random.normal(0, 0.018, n_samples).astype(np.float32)
         for f in factors.values()], axis=0
    )  # (12, n_samples)

    metadata = {
        'leads': list(factors.keys()), 'fs_hz': fs,
        'duration_sec': duration_sec, 'n_samples': n_samples,
        'units': 'mV', 'disease': disease, 'heart_rate_bpm': hr
    }
    return ecg_12, metadata

print("ECG generator ready.")

ECG generator ready.


In [4]:
def generate_segmentation(phenotype, img_size=128):
    edv     = phenotype['EDV_mL']
    wt      = phenotype['wall_thickness_mm']
    disease = phenotype.get('disease', 'NORMAL')

    r_lv = float(np.clip((edv * 600 / np.pi) ** (1/3), 14, 32))
    wt_px = float(np.clip(wt * 0.75, 3, 8))
    if disease == 'HCM':
        wt_px *= 1.5
    r_rv  = r_lv * 0.72
    cx, cy = img_size // 2, img_size // 2
    cx_rv  = cx + int(r_lv + r_rv * 0.85)

    Y, X    = np.ogrid[:img_size, :img_size]
    dist_lv = np.sqrt((X - cx)**2    + (Y - cy)**2)
    dist_rv = np.sqrt((X - cx_rv)**2 + (Y - cy)**2)

    seg = np.zeros((img_size, img_size), dtype=np.uint8)
    seg[(dist_lv >= r_lv) & (dist_lv < r_lv + wt_px)] = 2   # myocardium
    seg[dist_lv < r_lv]  = 1                                  # LV blood pool
    seg[dist_rv < r_rv]  = 3                                  # RV blood pool

    if disease == 'INFARCT':
        angle = np.arctan2(Y - cy, X - cx)
        scar  = (seg == 2) & (angle > np.pi/6) & (angle < np.pi/2)
        seg[scar] = 4   # scar label

    return seg

print("Segmentation generator ready.")

Segmentation generator ready.


In [5]:
def generate_coil_sensitivities(img_size=128, n_coils=8):
    cx, cy  = img_size // 2, img_size // 2
    Y, X    = np.ogrid[:img_size, :img_size]
    maps    = np.zeros((n_coils, img_size, img_size), dtype=np.complex64)
    angles  = np.linspace(0, 2*np.pi, n_coils, endpoint=False)
    radius  = img_size * 0.55
    for i, ang in enumerate(angles):
        cx_c = cx + radius * np.cos(ang)
        cy_c = cy + radius * np.sin(ang)
        dist  = np.sqrt((X - cx_c)**2 + (Y - cy_c)**2)
        mag   = np.exp(-dist**2 / (2*(img_size*0.38)**2)).astype(np.float32)
        phase = (ang * dist / img_size).astype(np.float32)
        maps[i] = mag * np.exp(1j * phase)
    # Normalize RSS to 1
    rss = np.sqrt(np.sum(np.abs(maps)**2, axis=0, keepdims=True)) + 1e-9
    maps = maps / rss
    return maps

def generate_vd_mask(img_size, R, acs_lines=None, seed=None):
    rng  = np.random.RandomState(seed)
    mask = np.zeros((img_size, img_size), dtype=bool)
    c    = img_size // 2
    
    # ACS lines: at most 25% of total needed lines
    total_needed = max(1, round(img_size / R))
    if acs_lines is None:
        acs_lines = max(4, min(24, total_needed // 4))
    
    h = acs_lines // 2
    mask[:, c-h:c+h] = True
    
    outer = [j for j in range(img_size) if not mask[0, j]]
    n_out = max(0, total_needed - acs_lines)
    n_out = min(n_out, len(outer))
    if n_out > 0:
        sel = rng.choice(outer, size=n_out, replace=False)
        mask[:, sel] = True
    return mask

def image_to_kspace(image, coil_maps, noise_std=0.005):
    n_coils = coil_maps.shape[0]
    kspace  = np.zeros_like(coil_maps, dtype=np.complex64)
    for c in range(n_coils):
        coil_img = image * coil_maps[c]
        kspace[c] = np.fft.fft2(coil_img).astype(np.complex64)
        if noise_std > 0:
            kspace[c] += (np.random.normal(0, noise_std, kspace[c].shape) +
                          1j * np.random.normal(0, noise_std, kspace[c].shape)).astype(np.complex64)
    return kspace

def rss_reconstruction(kspace, mask):
    n_coils = kspace.shape[0]
    kspace_us = kspace * mask[np.newaxis, :, :]
    images = np.zeros(kspace.shape[1:], dtype=np.complex64)
    for c in range(n_coils):
        images += np.abs(np.fft.ifft2(kspace_us[c]))**2
    return np.sqrt(images).real.astype(np.float32)

ACCEL_FACTORS = [4, 8, 10, 12]
print("K-space generators ready.")

K-space generators ready.


In [6]:
def md5_file(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest()

In [7]:
DISEASE_PARAMS = {
    'NORMAL':  dict(EDV=(130,170), ESV=(40,60),  EF=(58,68), WT=(7,10),  HR=(60,90)),
    'HCM':     dict(EDV=(110,150), ESV=(30,50),  EF=(62,75), WT=(14,20), HR=(55,85)),
    'DCM':     dict(EDV=(180,240), ESV=(100,150),EF=(30,45), WT=(6,8),   HR=(70,110)),
    'INFARCT': dict(EDV=(140,190), ESV=(70,110), EF=(40,55), WT=(7,11),  HR=(65,100)),
}

def sample_phenotype(disease, seed):
    rng = np.random.RandomState(seed)
    p   = DISEASE_PARAMS[disease]
    edv = float(rng.uniform(*p['EDV']))
    esv = float(rng.uniform(*p['ESV']))
    esv = min(esv, edv * 0.65)
    ef  = round((edv - esv) / edv * 100, 1)
    return {
        'disease':          disease,
        'EDV_mL':           round(edv, 1),
        'ESV_mL':           round(esv, 1),
        'EF_percent':       ef,
        'wall_thickness_mm':round(float(rng.uniform(*p['WT'])), 1),
        'heart_rate_bpm':   round(float(rng.uniform(*p['HR'])), 1),
        'generation_seed':  int(seed),
    }

print("Phenotype sampler ready.")

Phenotype sampler ready.


In [8]:
N_PATIENTS   = 500
N_PER_CLASS  = N_PATIENTS // len(DISEASE_PARAMS)   # 125 each
DISEASES     = list(DISEASE_PARAMS.keys())
IMG_SIZE     = 128
N_PHASES     = 20
N_COILS      = 8

all_phenotypes  = []
generation_log  = []
qc_passed       = 0
qc_failed       = 0

patient_id = 0
for disease in DISEASES:
    for idx in range(N_PER_CLASS):
        seed = patient_id * 100 + 7
        np.random.seed(seed)

        # --- Phenotype ---
        pheno = sample_phenotype(disease, seed)

        # --- QC: physiological constraints ---
        ef_calc = (pheno['EDV_mL'] - pheno['ESV_mL']) / pheno['EDV_mL'] * 100
        if not (15 <= pheno['EF_percent'] <= 85):
            qc_failed += 1
            generation_log.append({'patient_id': patient_id, 'status': 'FAIL_EF', 'disease': disease})
            patient_id += 1
            continue
        if pheno['EDV_mL'] <= pheno['ESV_mL']:
            qc_failed += 1
            generation_log.append({'patient_id': patient_id, 'status': 'FAIL_EDV_ESV', 'disease': disease})
            patient_id += 1
            continue

        # --- Cine phantom ---
        cine   = generate_cine_phantom(pheno, n_phases=N_PHASES, img_size=IMG_SIZE)

        # --- Segmentation (ED frame) ---
        seg    = generate_segmentation(pheno, img_size=IMG_SIZE)

        # --- ECG 12-lead ---
        ecg_arr, ecg_meta = generate_ecg_12lead(pheno)

        # --- Coil sensitivities ---
        coil_maps = generate_coil_sensitivities(img_size=IMG_SIZE, n_coils=N_COILS)

        # --- K-space (use ED frame = cine[0]) ---
        kspace_full = image_to_kspace(cine[0], coil_maps, noise_std=0.004)

        kspace_dict = {}
        for R in ACCEL_FACTORS:
            mask     = generate_vd_mask(IMG_SIZE, R, seed=seed+R)
            ks_us    = kspace_full * mask[np.newaxis, :, :]
            recon    = rss_reconstruction(kspace_full, mask)
            kspace_dict[R] = (ks_us, mask, recon)

        # --- Save files ---
        pid_dir = f'{OUTPUT_DIR}/patients/patient_{patient_id:04d}'
        os.makedirs(pid_dir, exist_ok=True)
        checksums = {}

        # phenotype.json
        ph_path = f'{pid_dir}/phenotype.json'
        with open(ph_path, 'w') as f: json.dump(pheno, f, indent=2)
        checksums['phenotype.json'] = md5_file(ph_path)

        # cine_reference.nii.gz  shape: (H, W, n_phases)
        cine_nii = nib.Nifti1Image(cine.transpose(1, 2, 0), affine=np.eye(4))
        cine_nii.header.set_zooms([1.5, 1.5, 8.0])
        cn_path = f'{pid_dir}/cine_reference.nii.gz'
        nib.save(cine_nii, cn_path)
        checksums['cine_reference.nii.gz'] = md5_file(cn_path)

        # segmentation.nii.gz
        seg_nii = nib.Nifti1Image(seg.astype(np.uint8), affine=np.eye(4))
        sg_path = f'{pid_dir}/segmentation.nii.gz'
        nib.save(seg_nii, sg_path)
        checksums['segmentation.nii.gz'] = md5_file(sg_path)

        # ecg_12lead.npz
        ecg_path = f'{pid_dir}/ecg_12lead.npz'
        np.savez_compressed(ecg_path, ecg=ecg_arr, leads=np.array(ecg_meta['leads']),
                            fs_hz=ecg_meta['fs_hz'], units='mV',
                            heart_rate_bpm=ecg_meta['heart_rate_bpm'])
        checksums['ecg_12lead.npz'] = md5_file(ecg_path)

        # coil_sensitivities.npz
        cs_path = f'{pid_dir}/coil_sensitivities.npz'
        np.savez_compressed(cs_path, coil_maps=coil_maps)
        checksums['coil_sensitivities.npz'] = md5_file(cs_path)

        # kspace_full.npz
        kf_path = f'{pid_dir}/kspace_full.npz'
        np.savez_compressed(kf_path, kspace=kspace_full)
        checksums['kspace_full.npz'] = md5_file(kf_path)

        # kspace_R{x}.npz for each acceleration
        for R, (ks_us, mask, recon) in kspace_dict.items():
            kr_path = f'{pid_dir}/kspace_R{R}.npz'
            np.savez_compressed(kr_path, kspace=ks_us, mask=mask,
                                R_nominal=R, acs_lines=24,
                                effective_R=round(mask.size/mask.sum(), 3))
            checksums[f'kspace_R{R}.npz'] = md5_file(kr_path)

        # provenance.json
        prov = {
            'patient_id': patient_id, 'disease': disease,
            'generation_seed': seed,
            'simulator': 'simplified_analytical_cardiac_phantom_v1.0',
            'ecg_model': 'parameterized_12lead_v1.0',
            'kspace_model': 'cartesian_vd_rss_v1.0',
            'img_size': IMG_SIZE, 'n_phases': N_PHASES, 'n_coils': N_COILS,
            'acceleration_factors': ACCEL_FACTORS,
            'qc_status': 'PASS',
            'checksums': checksums,
        }
        pv_path = f'{pid_dir}/provenance.json'
        with open(pv_path, 'w') as f: json.dump(prov, f, indent=2)

        all_phenotypes.append(pheno)
        qc_passed += 1
        generation_log.append({'patient_id': patient_id, 'status': 'PASS', 'disease': disease})

        if patient_id % 50 == 0:
            print(f"  Patient {patient_id:03d}/{N_PATIENTS} | {disease} | EF={pheno['EF_percent']}%")

        patient_id += 1

print(f"\nGeneration complete: {qc_passed} passed, {qc_failed} failed.")
with open(f'{OUTPUT_DIR}/generation_log.json', 'w') as f:
    json.dump(generation_log, f, indent=2)

  Patient 000/500 | NORMAL | EF=58.2%
  Patient 050/500 | NORMAL | EF=62.3%
  Patient 100/500 | NORMAL | EF=60.2%
  Patient 150/500 | HCM | EF=67.8%
  Patient 200/500 | HCM | EF=65.6%
  Patient 250/500 | DCM | EF=57.0%
  Patient 300/500 | DCM | EF=37.9%
  Patient 350/500 | DCM | EF=50.1%
  Patient 400/500 | INFARCT | EF=47.5%
  Patient 450/500 | INFARCT | EF=35.0%

Generation complete: 500 passed, 0 failed.


In [9]:
from collections import Counter

log_by_disease = defaultdict(lambda: {'PASS':0,'FAIL':0})
for entry in generation_log:
    log_by_disease[entry['disease']][entry['status'].split('_')[0]] += 1

pass_patients = [e for e in generation_log if e['status'] == 'PASS']

# Train/Val/Test split: 70/15/15 subject-level
np.random.seed(42)
splits_by_disease = {}
for disease in DISEASES:
    ids = [e['patient_id'] for e in pass_patients if e['disease'] == disease]
    np.random.shuffle(ids)
    n  = len(ids)
    n_tr = int(n * 0.70); n_vl = int(n * 0.15)
    splits_by_disease[disease] = {
        'train': ids[:n_tr], 'val': ids[n_tr:n_tr+n_vl], 'test': ids[n_tr+n_vl:]
    }

table_p1_1 = []
total_gen = total_pass = total_tr = total_vl = total_te = 0
for disease in DISEASES:
    gen  = log_by_disease[disease]['PASS'] + log_by_disease[disease]['FAIL']
    pss  = log_by_disease[disease]['PASS']
    sp   = splits_by_disease[disease]
    row  = {
        'Disease':        disease,
        'Generated':      gen,
        'QC_Accepted':    pss,
        'Acceptance_pct': round(pss/gen*100, 1) if gen > 0 else 0,
        'Train':          len(sp['train']),
        'Validation':     len(sp['val']),
        'Test':           len(sp['test']),
    }
    table_p1_1.append(row)
    total_gen += gen; total_pass += pss
    total_tr  += len(sp['train']); total_vl += len(sp['val']); total_te += len(sp['test'])

table_p1_1.append({
    'Disease':'TOTAL','Generated':total_gen,'QC_Accepted':total_pass,
    'Acceptance_pct':round(total_pass/total_gen*100,1) if total_gen>0 else 0,
    'Train':total_tr,'Validation':total_vl,'Test':total_te
})

print("\nTable P1-1: Released Cohort Composition")
print(f"{'Disease':<12} {'Generated':>10} {'QC_Pass':>8} {'Accept%':>8} {'Train':>7} {'Val':>6} {'Test':>6}")
print("-"*60)
for r in table_p1_1:
    print(f"{r['Disease']:<12} {r['Generated']:>10} {r['QC_Accepted']:>8} {r['Acceptance_pct']:>7.1f}% {r['Train']:>7} {r['Validation']:>6} {r['Test']:>6}")

with open(f'{OUTPUT_DIR}/tables/table_P1_1_cohort_composition.json', 'w') as f:
    json.dump(table_p1_1, f, indent=2)

# Save split manifest
split_manifest = {}
for d, sp in splits_by_disease.items():
    for role, ids in sp.items():
        for pid in ids:
            split_manifest[f'patient_{pid:04d}'] = {'disease': d, 'split': role}

with open(f'{OUTPUT_DIR}/split_manifest.json', 'w') as f:
    json.dump(split_manifest, f, indent=2)
print("\nSplit manifest saved.")


Table P1-1: Released Cohort Composition
Disease       Generated  QC_Pass  Accept%   Train    Val   Test
------------------------------------------------------------
NORMAL              125      125   100.0%      87     18     20
HCM                 125      125   100.0%      87     18     20
DCM                 125      125   100.0%      87     18     20
INFARCT             125      125   100.0%      87     18     20
TOTAL               500      500   100.0%     348     72     80

Split manifest saved.


In [10]:
# Cell: MD5 Checksums
import hashlib, json, os

DATA_ROOT = '/kaggle/working/qsyncardiotwin_v2/patients'
checksums = {}

for pid in range(500):
    pid_dir = os.path.join(DATA_ROOT, f'patient_{pid:04d}')
    checksums[f'patient_{pid:04d}'] = {}
    for fname in sorted(os.listdir(pid_dir)):
        fpath = os.path.join(pid_dir, fname)
        with open(fpath, 'rb') as f:
            md5 = hashlib.md5(f.read()).hexdigest()
        checksums[f'patient_{pid:04d}'][fname] = md5

checksum_path = '/kaggle/working/qsyncardiotwin_v2/checksums.json'
with open(checksum_path, 'w') as f:
    json.dump(checksums, f, indent=2)

print(f"Checksums saved for {len(checksums)} patients.")
print(f"Files per patient: {len(list(checksums.values())[0])}")
print("Sample patient_0000:")
for k,v in list(checksums['patient_0000'].items())[:4]:
    print(f"  {k}: {v[:16]}...")

Checksums saved for 500 patients.
Files per patient: 11
Sample patient_0000:
  cine_reference.nii.gz: 09ea690ab109edf5...
  coil_sensitivities.npz: cb7a0380daa72fd0...
  ecg_12lead.npz: b168d24f3260ccbe...
  kspace_R10.npz: 35ea7ff76265d543...


In [11]:
import numpy as np, os

DATA_ROOT = '/kaggle/working/qsyncardiotwin_v2/patients'
ecg_data = np.load(os.path.join(DATA_ROOT, 'patient_0000/ecg_12lead.npz'), allow_pickle=True)
print("Keys:", list(ecg_data.keys()))

Keys: ['ecg', 'leads', 'fs_hz', 'units', 'heart_rate_bpm']


In [12]:
# Cell: Table P1-2 — ECG Technical Validation
import numpy as np, os

DATA_ROOT = '/kaggle/working/qsyncardiotwin_v2/patients'

hr_list = []
fs_list = []

for pid in range(500):
    pid_dir = os.path.join(DATA_ROOT, f'patient_{pid:04d}')
    ecg_data = np.load(os.path.join(pid_dir, 'ecg_12lead.npz'), allow_pickle=True)
    hr_list.append(float(ecg_data['heart_rate_bpm']))
    fs_list.append(float(ecg_data['fs_hz']))

hr = np.array(hr_list)
fs = np.array(fs_list)

print("Table P1-2: ECG Technical Validation")
print(f"{'Feature':<25} {'Synthetic Mean±SD':<25} {'Real Reference':<25} {'In Range?'}")
print("-"*85)
print(f"{'Heart Rate (bpm)':<25} {hr.mean():.1f} ± {hr.std():.1f}{'':>10} {'60–100 bpm (Wagner 2020)':<25} {'✅' if 55<=hr.mean()<=110 else '❌'}")
print(f"{'Sampling Rate (Hz)':<25} {fs.mean():.0f} ± {fs.std():.0f}{'':>13} {'500 Hz standard':<25} {'✅' if fs.mean()==500 else '❌'}")
print(f"{'N patients':<25} {len(hr_list)}")
print(f"\nHR range: {hr.min():.1f} – {hr.max():.1f} bpm")
print(f"HR percentiles (25/50/75): {np.percentile(hr,25):.1f} / {np.percentile(hr,50):.1f} / {np.percentile(hr,75):.1f}")

# Per disease HR
import json
disease_hr = {}
for pid in range(500):
    pid_dir = os.path.join(DATA_ROOT, f'patient_{pid:04d}')
    with open(os.path.join(pid_dir, 'phenotype.json')) as f:
        pheno = json.load(f)
    d = pheno.get('disease', 'UNKNOWN')
    disease_hr.setdefault(d, [])
    disease_hr[d].append(hr_list[pid])

print("\nHR by disease:")
for d, vals in disease_hr.items():
    v = np.array(vals)
    print(f"  {d:<10}: {v.mean():.1f} ± {v.std():.1f} bpm")

Table P1-2: ECG Technical Validation
Feature                   Synthetic Mean±SD         Real Reference            In Range?
-------------------------------------------------------------------------------------
Heart Rate (bpm)          79.6 ± 12.2           60–100 bpm (Wagner 2020)  ✅
Sampling Rate (Hz)        500 ± 0              500 Hz standard           ✅
N patients                500

HR range: 55.0 – 109.1 bpm
HR percentiles (25/50/75): 70.7 / 78.8 / 87.9

HR by disease:
  NORMAL    : 75.7 ± 8.9 bpm
  HCM       : 70.1 ± 8.5 bpm
  DCM       : 88.8 ± 11.9 bpm
  INFARCT   : 83.9 ± 9.5 bpm


In [13]:
import json, os

DATA_ROOT = '/kaggle/working/qsyncardiotwin_v2/patients'
with open(os.path.join(DATA_ROOT, 'patient_0000/phenotype.json')) as f:
    pheno = json.load(f)
print(json.dumps(pheno, indent=2))

{
  "disease": "NORMAL",
  "EDV_mL": 133.1,
  "ESV_mL": 55.6,
  "EF_percent": 58.2,
  "wall_thickness_mm": 8.3,
  "heart_rate_bpm": 81.7,
  "generation_seed": 7
}


In [14]:
# Cell: Table P1-3 — CMR Anatomy & Function Validation
import numpy as np, json, os

DATA_ROOT = '/kaggle/working/qsyncardiotwin_v2/patients'

metrics = {'NORMAL': [], 'HCM': [], 'DCM': [], 'INFARCT': []}

for pid in range(500):
    pid_dir = os.path.join(DATA_ROOT, f'patient_{pid:04d}')
    with open(os.path.join(pid_dir, 'phenotype.json')) as f:
        pheno = json.load(f)
    d = pheno.get('disease', 'UNKNOWN')
    metrics[d].append({
        'EDV': pheno.get('EDV_mL'),
        'ESV': pheno.get('ESV_mL'),
        'EF':  pheno.get('EF_percent'),
        'WT':  pheno.get('wall_thickness_mm'),
    })

def stats(lst, key):
    v = np.array([x[key] for x in lst if x[key] is not None])
    return f"{v.mean():.1f} ± {v.std():.1f}"

print("Table P1-3: CMR Anatomy & Function Validation")
print(f"{'Measure':<15} {'NORMAL':<20} {'HCM':<20} {'DCM':<20} {'INFARCT':<20} {'Petersen Ref'}")
print("-"*105)
print(f"{'LVEDV (ml)':<15} {stats(metrics['NORMAL'],'EDV'):<20} {stats(metrics['HCM'],'EDV'):<20} {stats(metrics['DCM'],'EDV'):<20} {stats(metrics['INFARCT'],'EDV'):<20} 142±34 (M), 118±22 (F)")
print(f"{'LVESV (ml)':<15} {stats(metrics['NORMAL'],'ESV'):<20} {stats(metrics['HCM'],'ESV'):<20} {stats(metrics['DCM'],'ESV'):<20} {stats(metrics['INFARCT'],'ESV'):<20} 47±17 (M), 39±13 (F)")
print(f"{'LVEF (%)':<15} {stats(metrics['NORMAL'],'EF'):<20} {stats(metrics['HCM'],'EF'):<20} {stats(metrics['DCM'],'EF'):<20} {stats(metrics['INFARCT'],'EF'):<20} 63±5% (Normal)")
print(f"{'Wall Th (mm)':<15} {stats(metrics['NORMAL'],'WT'):<20} {stats(metrics['HCM'],'WT'):<20} {stats(metrics['DCM'],'WT'):<20} {stats(metrics['INFARCT'],'WT'):<20} 7–10 (N), 14–20 (HCM)")
print(f"\nN per class: 125 each | Reference: Petersen et al. JCMR 2017")

Table P1-3: CMR Anatomy & Function Validation
Measure         NORMAL               HCM                  DCM                  INFARCT              Petersen Ref
---------------------------------------------------------------------------------------------------------
LVEDV (ml)      150.5 ± 11.9         128.7 ± 11.8         211.0 ± 17.3         166.1 ± 13.5         142±34 (M), 118±22 (F)
LVESV (ml)      50.3 ± 5.7           39.7 ± 5.3           121.4 ± 12.1         88.5 ± 10.0          47±17 (M), 39±13 (F)
LVEF (%)        66.4 ± 4.6           69.0 ± 4.6           42.1 ± 6.8           46.4 ± 7.1           63±5% (Normal)
Wall Th (mm)    8.5 ± 0.9            16.9 ± 1.7           7.0 ± 0.6            9.1 ± 1.2            7–10 (N), 14–20 (HCM)

N per class: 125 each | Reference: Petersen et al. JCMR 2017


In [15]:
import numpy as np, os

DATA_ROOT = '/kaggle/working/qsyncardiotwin_v2/patients'

# Check R=8 mask for patient 0
ks = np.load(os.path.join(DATA_ROOT, 'patient_0000/kspace_R8.npz'))
print("Keys:", list(ks.keys()))
mask = ks['mask']
print("Mask shape:", mask.shape)
print("Sampled lines:", mask[0].sum())
print("Total lines:", mask.shape[1])
print("Effective R:", mask.shape[1] / mask[0].sum())

Keys: ['kspace', 'mask', 'R_nominal', 'acs_lines', 'effective_R']
Mask shape: (128, 128)
Sampled lines: 16
Total lines: 128
Effective R: 8.0


In [16]:
# Cell: Table P1-5 — MRI Physics & K-space Checks
import numpy as np, os

DATA_ROOT = '/kaggle/working/qsyncardiotwin_v2/patients'

rt_errors = []       # round-trip error
mask_rates = {4:[], 8:[], 10:[], 12:[]}

for pid in range(500):
    pid_dir = os.path.join(DATA_ROOT, f'patient_{pid:04d}')

    # Round-trip check: kspace_full -> ifft -> fft -> compare
    ks_full = np.load(os.path.join(pid_dir, 'kspace_full.npz'))['kspace']  # (n_coils, H, W)
    recon   = np.fft.ifft2(ks_full)
    ks_back = np.fft.fft2(recon)
    rt_err  = np.mean(np.abs(ks_back - ks_full)) / (np.mean(np.abs(ks_full)) + 1e-10)
    rt_errors.append(rt_err)

    # Mask check: actual vs nominal acceleration
    for R in [4, 8, 10, 12]:
        ks_r = np.load(os.path.join(pid_dir, f'kspace_R{R}.npz'))
        mask = ks_r['mask']   # (H, W)
        effective_R = mask.size / mask.sum()
        mask_rates[R].append(effective_R)

rt = np.array(rt_errors)
print("Table P1-5: MRI Physics & K-space Checks")
print(f"{'Check':<30} {'Mean':<15} {'Std':<10} {'Pass %'}")
print("-"*65)
print(f"{'Round-trip error':<30} {rt.mean():.2e}{'':>5} {rt.std():.2e}{'':>2} {(rt<1e-5).mean()*100:.1f}%")
for R in [4, 8, 10, 12]:
    mr = np.array(mask_rates[R])
    in_range = ((mr >= R*0.8) & (mr <= R*1.2)).mean()*100
    print(f"{'Effective R (nom='+str(R)+')':<30} {mr.mean():.2f}{'':>9} {mr.std():.2f}{'':>5} {in_range:.1f}%")

print(f"\nN = 500 patients | Round-trip tolerance: <1e-5")

Table P1-5: MRI Physics & K-space Checks
Check                          Mean            Std        Pass %
-----------------------------------------------------------------
Round-trip error               4.97e-07      1.86e-08   100.0%
Effective R (nom=4)            4.00          0.00      100.0%
Effective R (nom=8)            8.00          0.00      100.0%
Effective R (nom=10)           9.85          0.00      100.0%
Effective R (nom=12)           11.64          0.00      100.0%

N = 500 patients | Round-trip tolerance: <1e-5


In [17]:
# Cell: Table P1-4 — Cross-Modal Consistency
import numpy as np, json, os

DATA_ROOT = '/kaggle/working/qsyncardiotwin_v2/patients'

hr_list, edv_list, esv_list, ef_list, wt_list, disease_list = [], [], [], [], [], []

for pid in range(500):
    pid_dir = os.path.join(DATA_ROOT, f'patient_{pid:04d}')
    with open(os.path.join(pid_dir, 'phenotype.json')) as f:
        pheno = json.load(f)
    ecg = np.load(os.path.join(pid_dir, 'ecg_12lead.npz'), allow_pickle=True)
    hr_list.append(float(ecg['heart_rate_bpm']))
    edv_list.append(pheno['EDV_mL'])
    esv_list.append(pheno['ESV_mL'])
    ef_list.append(pheno['EF_percent'])
    wt_list.append(pheno['wall_thickness_mm'])
    disease_list.append(pheno['disease'])

hr  = np.array(hr_list)
edv = np.array(edv_list)
esv = np.array(esv_list)
ef  = np.array(ef_list)
wt  = np.array(wt_list)

# Pearson correlations: ECG HR vs CMR features
r_hr_edv = np.corrcoef(hr, edv)[0,1]
r_hr_esv = np.corrcoef(hr, esv)[0,1]
r_hr_ef  = np.corrcoef(hr, ef)[0,1]
r_hr_wt  = np.corrcoef(hr, wt)[0,1]

# Shuffled control
np.random.seed(42)
shuf = np.random.permutation(500)
r_shuf_edv = np.corrcoef(hr, edv[shuf])[0,1]
r_shuf_ef  = np.corrcoef(hr, ef[shuf])[0,1]

print("Table P1-4: Cross-Modal Consistency (ECG HR vs CMR features)")
print(f"{'Pair':<25} {'Matched r':<15} {'Shuffled r':<15} {'Delta'}")
print("-"*65)
print(f"{'HR vs EDV':<25} {r_hr_edv:>+.4f}{'':>8} {r_shuf_edv:>+.4f}{'':>8} {r_hr_edv-r_shuf_edv:>+.4f}")
print(f"{'HR vs ESV':<25} {r_hr_esv:>+.4f}")
print(f"{'HR vs EF':<25} {r_hr_ef:>+.4f}{'':>8} {r_shuf_ef:>+.4f}{'':>8} {r_hr_ef-r_shuf_ef:>+.4f}")
print(f"{'HR vs Wall Thickness':<25} {r_hr_wt:>+.4f}")

# Disease-conditional agreement
print("\nDisease-conditional HR mean (ECG-CMR agreement check):")
for d in ['NORMAL','HCM','DCM','INFARCT']:
    idx = [i for i,x in enumerate(disease_list) if x==d]
    print(f"  {d:<10}: HR={hr[idx].mean():.1f} bpm | EDV={edv[idx].mean():.1f} ml | EF={ef[idx].mean():.1f}%")

print(f"\nN = 500 | ECG and CMR derived from same latent phenotype → expect non-zero correlation")

Table P1-4: Cross-Modal Consistency (ECG HR vs CMR features)
Pair                      Matched r       Shuffled r      Delta
-----------------------------------------------------------------
HR vs EDV                 +0.5210         +0.0245         +0.4965
HR vs ESV                 +0.5590
HR vs EF                  -0.5096         -0.0495         -0.4602
HR vs Wall Thickness      -0.4806

Disease-conditional HR mean (ECG-CMR agreement check):
  NORMAL    : HR=75.7 bpm | EDV=150.5 ml | EF=66.4%
  HCM       : HR=70.1 bpm | EDV=128.7 ml | EF=69.0%
  DCM       : HR=88.8 bpm | EDV=211.0 ml | EF=42.1%
  INFARCT   : HR=83.9 bpm | EDV=166.1 ml | EF=46.4%

N = 500 | ECG and CMR derived from same latent phenotype → expect non-zero correlation


In [18]:
# Cell: Table P1-7 — Diversity & Privacy Audit
import numpy as np, json, os
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler

DATA_ROOT = '/kaggle/working/qsyncardiotwin_v2/patients'

# Build feature matrix
features, diseases = [], []
for pid in range(500):
    pid_dir = os.path.join(DATA_ROOT, f'patient_{pid:04d}')
    with open(os.path.join(pid_dir, 'phenotype.json')) as f:
        pheno = json.load(f)
    ecg = np.load(os.path.join(pid_dir, 'ecg_12lead.npz'), allow_pickle=True)
    features.append([
        float(ecg['heart_rate_bpm']),
        pheno['EDV_mL'], pheno['ESV_mL'],
        pheno['EF_percent'], pheno['wall_thickness_mm']
    ])
    diseases.append(pheno['disease'])

X = np.array(features)
scaler = StandardScaler()
X_sc = scaler.fit_transform(X)

# --- Membership Inference Attack (MI AUC) ---
# Train set = label 1, synthetic "non-members" = label 0 (shuffled features)
np.random.seed(42)
with open('/kaggle/working/qsyncardiotwin_v2/split_manifest.json') as f:
    manifest = json.load(f)

train_ids = [int(k.split('_')[1]) for k,v in manifest.items() if v['split']=='train']
test_ids  = [int(k.split('_')[1]) for k,v in manifest.items() if v['split']=='test']

X_train = X_sc[train_ids]
X_test  = X_sc[test_ids]

X_mi = np.vstack([X_train, X_test])
y_mi = np.array([1]*len(X_train) + [0]*len(X_test))

clf = LogisticRegression(max_iter=1000, random_state=42)
cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
from sklearn.metrics import roc_auc_score
auc_scores = cross_val_score(clf, X_mi, y_mi, cv=cv, scoring='roc_auc')
mi_auc = auc_scores.mean()

# --- Nearest-neighbor distance (diversity) ---
from sklearn.metrics import pairwise_distances
nn_dists = []
D = pairwise_distances(X_sc, metric='euclidean')
np.fill_diagonal(D, np.inf)
nn_dists = D.min(axis=1)

# --- Coverage: std of each feature ---
coverage = X.std(axis=0)
feat_names = ['HR','EDV','ESV','EF','WT']

print("Table P1-7: Diversity & Privacy Audit")
print(f"\nMembership Inference AUC: {mi_auc:.4f} ± {auc_scores.std():.4f}")
print(f"  (ideal random = 0.50 | threshold < 0.60 = low risk)")
print(f"  {'✅ LOW RISK' if mi_auc < 0.60 else '⚠️ CHECK NEEDED'}")

print(f"\nNearest-Neighbor Distance (diversity):")
print(f"  Mean: {nn_dists.mean():.4f} | Std: {nn_dists.std():.4f} | Min: {nn_dists.min():.4f}")

print(f"\nFeature Coverage (std across 500 patients):")
for n, s in zip(feat_names, coverage):
    print(f"  {n}: {s:.2f}")

print(f"\nDuplicate rate: {(nn_dists < 0.01).mean()*100:.1f}%")
print(f"N = 500 patients")

Table P1-7: Diversity & Privacy Audit

Membership Inference AUC: 0.5198 ± 0.0742
  (ideal random = 0.50 | threshold < 0.60 = low risk)
  ✅ LOW RISK

Nearest-Neighbor Distance (diversity):
  Mean: 0.3113 | Std: 0.1131 | Min: 0.0347

Feature Coverage (std across 500 patients):
  HR: 12.17
  EDV: 33.16
  ESV: 33.56
  EF: 13.24
  WT: 4.00

Duplicate rate: 0.0%
N = 500 patients
